# SLM Chess Capability Benchmark @ NeurIPS 2026 — CHECK MODE (tiny, raises on failure)

Paired win/lose small-model chess study (see README). Results land in `results_check/` and are zipped for download.
- Positions + exact oracles: committed (`data/positions/`), generated once by `scripts/generate_positions.py`.
- Engine + dataset tests gate every run: `scripts/test_engine.py`.
- Sweep: `scripts/run_suite.py` (models x tasks x {{win,lose}}).

## 1. Get the repo (GitHub secret method)

The repo is **private**. On Kaggle, a secret reaches the notebook ONLY if it is **attached to this notebook** and the kernel is started AFTER attaching:

1. Notebook editor -> **+ Add** (top-right) -> **Add secret** -> select `GITHUB_TOKEN` (it must exist under Account settings -> Secrets; value = a classic PAT with `repo` scope).
2. **Save** the notebook (Ctrl+S).
3. **Kernel -> Restart & Run All** (env vars are injected at kernel start; plain "Run All" does NOT pick up newly attached secrets).

This cell reads the token from the env var, and falls back to Kaggle's own `kaggle_secrets` API if the env var is missing.

In [1]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "neuro-symbolic-pathfinding"
# ALWAYS start from a fresh clone: re-runs in the same Kaggle session keep the
# old /kaggle/working repo, and stale code has bitten us more than once.
if REPO.exists():
    shutil.rmtree(REPO)

def find_token():
    for name in ("GITHUB_TOKEN", "GH_TOKEN"):
        if os.environ.get(name):
            return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None

# diagnostic: what token-ish env vars are actually present?
present = sorted(k for k in os.environ if "TOKEN" in k.upper() or "SECRET" in k.upper())
print("token-ish env vars present:", present, flush=True)
token = find_token()
print("GITHUB_TOKEN resolved:", bool(token), flush=True)
url = "https://github.com/Vedang-P/neuro-symbolic-pathfinding.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
res = subprocess.run(["git", "clone", "--quiet", url, str(REPO)],
                     capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError(
        "git clone failed. The token did not reach this run. Fix order: "
        "(1) + Add -> Add secret -> GITHUB_TOKEN; (2) SAVE the notebook; "
        "(3) Kernel -> Restart & Run All. Then check the diagnostic line above: "
        "if 'token-ish env vars present' is empty, the secret is not attached to "
        "THIS notebook. Stderr: " + res.stderr[-300:]
    )
os.chdir(REPO)
print("cwd:", Path.cwd())

token-ish env vars present: ['KAGGLE_API_V1_TOKEN', 'KAGGLE_DATA_PROXY_TOKEN', 'KAGGLE_USER_SECRETS_TOKEN']
GITHUB_TOKEN resolved: True
cwd: /kaggle/working/neuro-symbolic-pathfinding


## 2. Dependencies (forced upgrade; transformers must be >= 5.13 for Gemma 4)

In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import transformers
if int(transformers.__version__.split(".")[0]) < 5:
    raise RuntimeError(
        f"transformers {transformers.__version__} is too old for Gemma 4 "
        "(needs >= 5.13). The upgrade failed — check the pip install output above."
    )
print("transformers", transformers.__version__, "(gemma4 support OK)")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


transformers 5.14.1 (gemma4 support OK)
torch 2.10.0+cu128 cuda True Tesla T4


## 3. Stage runner (never raises; the verdict cell checks results)

In [3]:
import json, time, shutil
from pathlib import Path

STAGE_LOG = Path("results_check/stage_log.json")
def run_stage(name, args, timeout_min):
    Path("results_check").mkdir(parents=True, exist_ok=True)
    rec = {"stage": name, "status": "running", "elapsed_min": None}
    t0 = time.time()
    try:
        res = subprocess.run(args, timeout=timeout_min * 60)
        rec["status"] = "ok" if res.returncode == 0 else "failed"
        rec["returncode"] = res.returncode
    except subprocess.TimeoutExpired:
        rec["status"] = "timeout"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = str(e)[:200]
    rec["elapsed_min"] = round((time.time() - t0) / 60, 1)
    entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
    entries.append(rec)
    STAGE_LOG.write_text(json.dumps(entries, indent=1))
    print(f"stage {name}: {rec['status']} ({rec['elapsed_min']}min)", flush=True)
    return rec["status"]

## 4. Gate: engine + dataset tests

In [4]:
status = run_stage("engine_tests", [sys.executable, "scripts/test_engine.py", "--quick"], 10)
if status != "ok":
    raise RuntimeError("engine tests failed -- see output above")

ok   sq_to_algebraic a1
ok   algebraic_to_sq h8
ok   algebraic_to_sq junk
ok   algebraic_to_sq short
ok   8x8 mate-in-1 Qg7-style
ok   3x3 KvK 2 legal moves
ok   value KvK draw
ok   value 3x3 KQvK win
ok   value 5x5 KQvK win
ok   value blocked pawn draw
ok   value bK can capture, draw
ok   value far pawn promotes, win
ok   value KQvKQ 3x3 draw
ok   win moves subset of legal
ok   lose moves subset of legal
ok   win and lose disjoint
ok   win position has win moves
ok   win position has non-win moves
ok   dataset bestmove-8x8 non-empty
ok   bm-0000 has oracle data
ok   bm-0000 has best_move
ok   bm-0000 best_move legal in our variant
ok   bm-0000 bestmove non-vacuous
ok   bm-0001 has oracle data
ok   bm-0001 has best_move
ok   bm-0001 best_move legal in our variant
ok   bm-0001 bestmove non-vacuous
ok   bm-0002 has oracle data
ok   bm-0002 has best_move
ok   bm-0002 best_move legal in our variant
ok   bm-0002 bestmove non-vacuous
ok   bm-0003 has oracle data
ok   bm-0003 has best_move
ok

## 5. Position generation sanity (tiny, exercises the oracle path)

In [5]:
status = run_stage("gen_check", [sys.executable, "scripts/generate_positions.py", "--check", "--out", "results_check/positions"], 10)
if status != "ok":
    raise RuntimeError("position generation check failed")

sm-3x3-win: 3 positions 0.0s
sm-3x3-draw: 3 positions 0.3s
sm-5x5-win: 3 positions 3.9s
sm-5x5-draw: 3 positions 19.7s
mate1-8x8: 3 positions 0.0s
mob-8x8: 3 positions 0.0s
total 24.0s -> results_check/positions
stage gen_check: ok (0.4min)


## 6. Recover completed results (after a died session)

Every completed cell's summary is backed up to the public live repo by `--monitor` (check results and full results live in separate namespaces, so recovery can never mix them). If this is a fresh session, pull them back so `--resume` can skip what already ran.

In [6]:
import json, urllib.request
from pathlib import Path

out = Path("results_check/chess")
out.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/Vedang-P/chess-bench-live/main"
idx_url = f"{base}/results_check/index.json"
if list(out.glob("*.summary.json")):
    print("results already present locally")
else:
    try:
        idx = json.load(urllib.request.urlopen(idx_url, timeout=20))
        for name in idx["files"]:
            url = f"{base}/results_check/chess/{name}"
            (out / name).write_bytes(urllib.request.urlopen(url, timeout=20).read())
        print(f"recovered {len(idx['files'])} completed summaries from the live repo")
    except Exception as e:
        print("nothing to recover (first run or no backup yet):", e)

recovered 17 completed summaries from the live repo


## 7. The chess sweep (models x tasks, paired win/lose)

`--monitor` publishes live progress + per-cell result backups to the public dashboard repo (monitor/state.json, results/*). `--resume` skips cells whose summary already exists (recovered in the previous cell).

In [ ]:
sweep_args = [sys.executable, "scripts/run_suite.py", "--output_dir", "results_check/chess",
              "--monitor", "--monitor-interval", "120"]
if True:
    sweep_args.append("--check")
sweep_args.append("--resume")   # skip cells whose summaries were recovered
status = run_stage("chess_sweep", sweep_args, 50)
print("sweep:", status)

suite: 6 models x 6 task-variant cells (CHECK mode, resume)
  resume: deepseek-r1-distill-qwen-1.5b x mate1-lichess:grid already done — loaded 1 rows
  resume: deepseek-r1-distill-qwen-1.5b x mate1-lichess:fen already done — loaded 1 rows
  resume: deepseek-r1-distill-qwen-1.5b x mate2-lichess:grid already done — loaded 1 rows
  resume: deepseek-r1-distill-qwen-1.5b x mate2-lichess:fen already done — loaded 1 rows
  resume: deepseek-r1-distill-qwen-1.5b x bestmove-8x8:grid already done — loaded 1 rows
  resume: deepseek-r1-distill-qwen-1.5b x bestmove-8x8:fen already done — loaded 1 rows
  resume: smollm2-1.7b x mate1-lichess:grid already done — loaded 1 rows
  resume: smollm2-1.7b x mate1-lichess:fen already done — loaded 1 rows
  resume: smollm2-1.7b x mate2-lichess:grid already done — loaded 1 rows
  resume: smollm2-1.7b x mate2-lichess:fen already done — loaded 1 rows
  resume: smollm2-1.7b x bestmove-8x8:grid already done — loaded 1 rows
  resume: smollm2-1.7b x bestmove-8x8:fen a

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 266.18it/s]


  [bestmove-8x8 qwen2.5-1.5b fen] 1/1 (1.4s/position)
{
 "conditions": {
  "win": {
   "n": 1,
   "no_answer": 0,
   "parse_error": 0,
   "illegal": 1,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 1.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 }
}
    36s

>>> qwen2.5-3b x mate1-lichess:grid (n=1, tokens=1024)


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 191.60it/s]


  [mate1-lichess qwen2.5-3b grid] 1/1 (1.8s/position)
{
 "conditions": {
  "win": {
   "n": 1,
   "no_answer": 0,
   "parse_error": 1,
   "illegal": 0,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 0.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    37s

>>> qwen2.5-3b x mate1-lichess:fen (n=1, tokens=1024)


...

Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 515.77it/s]


  [mate1-lichess gemma4-e2b grid] 1/1 (104.2s/position)
{
 "conditions": {
  "win": {
   "n": 1,
   "no_answer": 0,
   "parse_error": 1,
   "illegal": 0,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 0.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    165s

>>> gemma4-e2b x mate1-lichess:fen (n=1, tokens=1024)


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 508.11it/s] 


  [mate1-lichess gemma4-e2b fen] 1/1 (102.8s/position)
{
 "conditions": {
  "win": {
   "n": 1,
   "no_answer": 0,
   "parse_error": 1,
   "illegal": 0,
   "legal": 0,
   "compliant": 0,
   "noncompliant": 0,
   "undefined": 0,
   "compliance_of_legal": null,
   "parse_rate": 0.0,
   "legal_rate": 0.0,
   "compliance_strict": 0.0
  }
 },
 "divergence_rate": null
}
    124s

>>> gemma4-e2b x mate2-lichess:grid (n=1, tokens=1024)


Loading weights: 100%|██████████| 1951/1951 [00:03<00:00, 540.58it/s] 


## 8. Results table

In [ ]:
import pandas as pd
csv_path = Path("results_check/chess/comparison_table.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df)
    print("rows:", len(df))
else:
    print("no comparison table -- sweep did not complete")

## 9. Verdict (check mode: fail loudly)

In [ ]:
entries = json.loads(STAGE_LOG.read_text()) if STAGE_LOG.exists() else []
fails = [e for e in entries if e["status"] != "ok"]
if fails:
    raise RuntimeError(f"check mode: {len(fails)} failed stages: {[e['stage'] for e in fails]}")
print("ALL CHECK STAGES PASSED")

## Notes
- **Getting the repo (secret method):** the `GITHUB_TOKEN` secret must be attached to this notebook (+ Add -> Add secret), the notebook SAVED, and the kernel RESTARTED -- env vars are injected at kernel start only.
- **Resume after a died session:** re-run the notebook with a trimmed sweep, e.g. `run_suite.py --models <remaining> --tasks <remaining> --output_dir results/chess`; per-run JSONs under `results/chess/*.summary.json` are the source of truth; the CSV is rebuilt at the end.
- **Gemma models** need the `HF_TOKEN` Kaggle secret (gated access).
- **Timeouts:** full-mode sweep is capped at 12h; typical T4 estimate ~1-2 min/position-cell, well under a single Kaggle session.